# Tutorial 16 — Synthetic Digital Image Correlation

Known-ground-truth image deformation and transparent local DIC.

In [ ]:
LANGUAGE = "en"
from pathlib import Path
import sys

def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found")

REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))
import biomechanics_tutorials
print(Path(biomechanics_tutorials.__file__).resolve())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from biomechanics_tutorials.digital_image_correlation import (
    DICParameters, SpeckleParameters, generate_speckle_pattern,
    displacement_field, warp_image, run_subset_dic,
    sample_truth_on_dic_grid, dic_metrics, strain_fields,
    export_dic_dataset,
)

In [ ]:
shape = (128, 144)
reference = generate_speckle_pattern(shape, SpeckleParameters(seed=16, density=0.035))
u_true, v_true = displacement_field(shape, "heterogeneous", amplitude=4.0)
deformed = warp_image(reference, u_true, v_true, interpolation_order=3, noise_std=0.006, seed=2)
parameters = DICParameters(subset_radius=11, step=14, search_radius=6)
result = run_subset_dic(reference, deformed, parameters, correlation_threshold=0.45)
u_grid = sample_truth_on_dic_grid(u_true, result)
v_grid = sample_truth_on_dic_grid(v_true, result)
metrics = dic_metrics(result.u, result.v, u_grid, v_grid, result.valid)
metrics

In [ ]:
titles = {
    "en": ["Reference", "Deformed", "Recovered u", "Vector error"],
    "ru": ["Исходное", "Деформированное", "Восстановленное u", "Векторная ошибка"],
}[LANGUAGE]
error = np.hypot(result.u - u_grid, result.v - v_grid)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(reference, cmap="gray")
axes[1].imshow(deformed, cmap="gray")
axes[2].imshow(result.u, cmap="coolwarm")
axes[3].imshow(error, cmap="magma")
for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
yy, xx = np.indices(shape, dtype=float)
u_affine = 0.035 * xx + 0.012 * yy
v_affine = -0.006 * xx - 0.015 * yy
strain = strain_fields(u_affine, v_affine)
{
    "median_small_exx": float(np.median(strain.small_exx)),
    "median_small_eyy": float(np.median(strain.small_eyy)),
    "median_green_exx": float(np.median(strain.green_exx)),
    "median_jacobian": float(np.median(strain.jacobian)),
}

In [ ]:
output_path = REPOSITORY_ROOT / "tutorials/16-synthetic-digital-image-correlation/data/notebook_dic_dataset.npz"
export_dic_dataset(output_path, reference, deformed, u_true, v_true, {"seed": 16, "source": "Tutorial 16 notebook"})
output_path